# Runner v1.1 (Triều) — train 1 click trên Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quocluongg/d09-eval-colab/blob/main/colab/run_v1_1.ipynb)

**Cách chạy:** `Runtime` → `Run all` (Ctrl+F9). Không phải chỉnh sửa gì thêm.

Pipeline làm đúng các lệnh trong `e3_send_to_quoc.md` của Triều:

1. Kiểm tra số dòng: `data/features.csv` = **38465**, `selfcheck_v1/data/features.csv` = **45318** (lệch là dừng).
2. Tạo venv + cài đúng `requirements.lock.txt` (17 pin).
3. `selfcheck --data selfcheck_v1/data` — **chạy trên data v1**, không chạy trên `data` v2 (reference là của v1).
4. `run --config config_tuned/experiment.json --data data --out out --models logistic_regression random_forest xgboost isolation_forest mlp autoencoder xgboost_if_feature combo_rank_mean combo_or`
5. `pack --out out`
6. In báo cáo + tải `out.zip` về máy.

Lệnh nào lỗi thì pipeline **dừng ngay** và in nguyên văn lỗi, không tự sửa. Log nằm trong `runner_v1_1/logs/`.

**Python:** Colab có `python3.12` (3.12.3) nhưng **không kèm pip**, nên script chọn interpreter có pip (python của Colab) và in version thực tế dùng để chạy. Triều nói 3.13 cho cùng số ở package trước, nhưng vẫn phải gửi lại text selfcheck — notebook in `python3 --version` để bạn nhắn Triều.

Không sửa: `experiment.json`, `experiment.sha256`, `config_tuned/`, `requirements.lock.txt`, `reference/`, `data/`, `selfcheck_v1/`, `src/`, `tests/`.

In [ ]:
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/quocluongg/d09-eval-colab.git"
REPO_DIR = pathlib.Path("/content/d09-eval-colab")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("cwd    =", os.getcwd())
print("commit =", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("python =", sys.version.split()[0])
print("python3 --version =", subprocess.run(["python3", "--version"], capture_output=True, text=True).stdout.strip())

In [ ]:
import subprocess

proc = subprocess.Popen(
    ["bash", "colab/run_v1_1.sh"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"run_v1_1.sh failed with exit code {rc} — dừng, không tự sửa. Đọc lỗi nguyên văn ở trên và trong runner_v1_1/logs/.")

In [ ]:
from pathlib import Path

print(Path("runner_v1_1/logs/RESULTS.txt").read_text(encoding="utf-8"))

from google.colab import files

files.download("runner_v1_1/out.zip")

## Gửi lại Triều đủ 5 món (What to Send Back)

1. `out.zip` do lệnh `pack` tạo — **không giải nén, không đổi tên**.
2. SHA-256 của zip: dòng `2_zip_sha256=` trong `runner_v1_1/logs/RESULTS.txt`.
3. Nguyên văn toàn bộ output `selfcheck` — mục `=== SELFCHECK (verbatim) ===`.
4. Máy: `4_os=`, `4_ram_total_gb=` — mục `=== MACHINE ===`.
5. Tổng thời gian chạy `run`: `5_run_wall_seconds=` / `5_run_wall_hms=`.

Kèm thêm: `python3 --version` (Colab = 3.13, không có 3.12.10) như Triều dặn.